RADIUS sta per **Remote Authentication Dial-In User Service**. Il nome deriva dall'epoca in cui gli utenti si collegavano ad internet tramite modem e linee telefoniche, il suo utilizzo si è poi esteso ad altri tipi di accesso alla rete. 

Il problema da risolvere è la gestione degli utenti in una rete con molti punti di accesso: immaginiamo un operatore che ha molti dispositivi attraverso cui gli utenti possono connettersi --> gestire separatamente su ogni dispositivo password, autorizzazioni e configurazioni è complicato, ogni modifica dovrebbe essere replicata su tutti i dispositivi.

RADIUS permette di **centralizzare le decisioni di accesso**, mantenendo distribuiti i dispositivi che collegano concretamente gli utenti alla rete.  
I partecipanti sono:
- **Utente finale** == chi vuole accedere alla rete, presenta le proprie credenziali al dispositivo di accesso.
- **NAS** (Network Access Server) == gestisce l'accesso dell'utente e applica la decisione del server
- **Server RADIUS** == riceve le richieste di accesso dal NAS, verifica le credenziali e stabilisce quali servizi autorizzare

La cosa importante da capire è che **il client che comunica con il server RADIUS non è l'utente finale, ma il NAS**! A sua volta il NAS comunica con l'utente finale però tramite il protocollo previsto dal sistema di accesso (es. PPP), mentre con il server RADIUS comunica tramite il protocollo RADIUS.

In generale il funzionamento è il seguente:
1. L'utente presenta al NAS le informazioni necessarie per autenticarsi
2. il NAS le inserisce in una richiesta RADIuS e le invia al server RADIUS
3. il server RADIUS verifica le credenziali e risponde al NAS con un messaggio di accettazione o rifiuto
4. il NAS applica la decisione e informa l'utente

RADIUS è un protocollo che fornisce le funzionalità **AAA**: **Authentication** (l'utente è davvero chi dice di essere? es. verifica credenziali), **Authorization** (quali servizi può utilizzare? es. controlla le autorizzazioni) e **Accounting** (quali risorse ha utilizzato? es. durata sessione, traffico, info per la fatturazione).

RADIUS classico è protocollo client server basato su UDP/IP, con porta 1812 per l'autenticazione. La centralizzazione è logica: possono chiaramente esserci server secondari/repliche.

La richiesta NAS prende il nome di **Access-Request**. Le risposte principali sono *Access-Accept* (accesso autorizzato), *Access-Reject* (accesso negato) e Access-Challenge (richiesta di ulteriori informazioni).

Access-Challenge non indica però necessariamente challenge crittografica: infatti alcuni metodi richiedono più messaggi, come quelli che trasportano uno scambio TLS (come vedremo).

In generale un server RADIUS può inoltrare la richiesta ad un altro server ancora, comportandosi quindi come **proxy**.  
Ciò è utile ad es. nel roaming: l'utente accede attraverso la rete ospitante, ma le sue credenziali devono essere verificate dal server del suo provider.  
Per i nostri scopi è sufficiente capire che il NAS può ricevere una decisione ottenuta da più server, concentreremo l'analisi di sicurezza però sul collegamento diretto tra NAS e server RADIUS.

Il server dispone di informazioni relative a **utenti** (credenziali, metodo di autenticazione, autorizzazioni...), **client RADIUS** (NAS autorizzati a comunicare col server), **accounting** (info sull'utilizzo dei servizi)

### Funzionamento RADIUS
Due meccanismi principali per il RADIUS originario:
- **Autenticazione delle risposte del server**, mediante segreto condiviso
- **Protezione della password dell'utente**, mediante meccanismo specifico di RADIUS

Prima di studiarli, distinguiamo i due segreti:
$$P_U = \text{password dell'utente} \quad S = \text{segreto condiviso tra NAS e server RADIUS}$$
La password permette di autenticare l'utente, mentre il segreto $S$ serve a proteggere la comunicazione tra NAS e server RADIUS. **L'utente non deve conoscere il segreto $S$**.

Tra i problemi del protocollo RADIUS originario:
- le risposte del server sono autenticate, mentre dalla parte del NAS (gli Access-Request) no
- l'autenticazione della risposta usa una costruzione basata direttamente su MD5 invece che su HMAC-MD5
- MD5 è fissato dal protocollo, quindi non è possibile sostituirlo con un algoritmo più sicuro
- lo stesso segreto viene impiegato anche per proteggere la password

Nel modello che considereremo si assume che **il NAS e il server RADIUS siano fidati**, che **l'utente possa essere attaccante** e **possa intercettare e modificare i messaggi tra NAS e server RADIUS**.  
Il fatto che NAS sia assunto fidato è necessario perché è il dispositivo che poi applica la decisione --> se fosse compromesso potrebbe consentire l'accesso a chiunque.

La preoccupazione che ci interessa riguarda le risposte del server: se il server risponde con *Access-Reject* e un attaccante riesce a trasformare il messaggio in *Access-Accept* allora è nu burdel.  
Inoltre le risposte del server contengono attributi che specificano le autorizzazioni dell'utente, quindi non basta proteggere il "sì" o il "no", ma anche quali servizi vengono concessi.

### Autenticazione della risposta RADIUS
Siano:
$$I = \text{Identifier della richiesta}, \quad A = \text{Request-Authenticator}$$
Allora il NAS genera $A$ come valore casuale a 16 byte e lo inserisce nell'Access-Request.  
In un Access-Request, $A$ **svolge il ruolo di nonce** (il nome confonde, da solo questo campo non è MAC della richiesta).  
Il NAS conserva localmente le info necessarie ad associare la richiesta alla sua risposta, incluso il valore $A$. 

Quando il server riceve la richiesta, prende la decisione e calcola un tag:
$$t = MD5(\text{Code} || I || \text{Length} || A || \text{Attributes} || S)$$
I campi Code, Identifier, Length e Attributes sono quelli della risposta. $A$ è invece il Request-Authenticator della richiesta originale.

Il tag viene inserito nel campo Authenticator della risposta, che in questo caso prende il nome di **Response-Authenticator**. Quindi il campo Authenticator assume due significati nel protocollo RADIUS in base al pacchetto:
- se pacchetto Access-Request --> valore casuale usato come nonce
- se pacchetto Access-Accept/Reject/Challenge --> tag di autenticazione della risposta

Quando il NAS riceve la risposta:
1. individua la richiesta corrispondente e recupera il nonce $A$ conservato
2. ricalcola il tag $t$ usando la risposta ricevuta e il segreto $S$
3. confronta il risultato con il Response-Authenticator

Se i valori coincidono --> verifica riuscita

**Perché serve il segreto?** Per impedire all'attaccante di costruire liberamente un tag valido per una risposta modificata **(autenticazione del messaggio --> protezione da falsificazioni e modifiche i.e. integrità)**
**Perché serve il nonce?** Per collegare la risposta alla richiesta corrente. Senza il nonce l'attaccante potrebbe ripresentare una vecchia risposta positiva **(nonce e gestione dello stato --> protezione da replay)**

**Perché è il NAS a scegliere il nonce?** Perché è il NAS a dover verificare la risposta, quindi è lui che deve poterla associare alla richiesta corrente. Se fosse il server a scegliere il nonce --> il NAS dovrebbe disporre di un ulteriore meccanismo per stabilire se la risposta è nuova. La presenza di molti NAS rende questo meccanismo difficile. 

Il limite di questa protezione è che sì, copre la risposta e il nonce della richiesta, **ma non copre tutti gli attributi della richiesta Access-Request**: questa distinzione sarà il punto dell'attacco a CHAP (una risposta autentica e fresca può essere la risposta a una richiesta che un attaccante ha modificato!)

SLIDE 11-13